# BGE + Milvus RAG on a Single Document

Converts one document with Docling, chunks it with HierarchicalChunker, embeds the chunks with BGE-large into Milvus Lite and answers a question with Gemini.

| `SOURCE` | Document | Extra |
|---|---|---|
| `"excel"` | `docs/sample_docs/sample_statement.xlsx` | also runs the batch of form questions (last three cells) |
| `"pdf"` | `docs/SFS XBRL PDF_53-54.pdf` | – |

In [ ]:
# Document to index: "excel" or "pdf" (see the table above)
SOURCE = "excel"

In [ ]:
SOURCE_SETTINGS = {
    "excel": {"path": "docs/sample_docs/sample_statement.xlsx",
              "question": "Mr. Director A's number of Shares as of March 31 2024",
              "form_questions": True},
    "pdf":   {"path": "docs/SFS XBRL PDF_53-54.pdf",
              "question": "Have the  The financial statements have been prepared in accordance with Indian Accounting Standards?",
              "form_questions": False},
}
if SOURCE not in SOURCE_SETTINGS:
    raise ValueError(f"Unknown SOURCE {SOURCE!r}; choose one of {sorted(SOURCE_SETTINGS)}")
SOURCE_PATH = SOURCE_SETTINGS[SOURCE]["path"]
QUESTION = SOURCE_SETTINGS[SOURCE]["question"]
RUN_FORM_QUESTIONS = SOURCE_SETTINGS[SOURCE]["form_questions"]
print(f"Source: {SOURCE} -> {SOURCE_PATH}")

In [ ]:
import torch

# Check if GPU or MPS is available
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"CUDA GPU is enabled: {torch.cuda.get_device_name(0)}")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
    print("MPS GPU is enabled.")
else:
    raise OSError(
        "No GPU or MPS device found. Please check your environment and ensure GPU or MPS support is configured."
    )

In [ ]:
from sentence_transformers import SentenceTransformer

# Load BGE embedding model from Hugging Face
embedding_model = SentenceTransformer('BAAI/bge-large-en-v1.5', device='cuda')

In [ ]:
#Load embedding model
def emb_text(text):
    embedding = embedding_model.encode(text, normalize_embeddings=True)
    return embedding.tolist()

In [ ]:
#Test snippet
test_embedding = emb_text("This is a test")
embedding_dim = len(test_embedding)
print(f"Embedding dimension: {embedding_dim}")
print(f"First 10 values: {test_embedding[:10]}")

In [ ]:
from docling_core.transforms.chunker import HierarchicalChunker
from docling.document_converter import DocumentConverter
from pymilvus import MilvusClient
from tqdm import tqdm
import json

converter = DocumentConverter()
chunker = HierarchicalChunker()

# Convert the input file to Docling Document
source = SOURCE_PATH
doc = converter.convert(source).document

# Perform hierarchical chunking
texts = [chunk.text for chunk in chunker.chunk(doc)]

In [ ]:
milvus_client = MilvusClient(uri="./milvus_pdf.db")
collection_name = "rag_pdf"

In [ ]:
data = []
if milvus_client.has_collection(collection_name):
    milvus_client.drop_collection(collection_name)

milvus_client.create_collection(
    collection_name=collection_name,
    dimension=embedding_dim,
    metric_type="IP",  # Inner product distance
    consistency_level="Strong",  # Supported values are (`"Strong"`, `"Session"`, `"Bounded"`, `"Eventually"`). See https://milvus.io/docs/consistency.md#Consistency-Level for more details.
)
for i, chunk in enumerate(tqdm(texts, desc="Processing chunks")):
    embedding = emb_text(chunk)
    data.append({"id": i, "vector": embedding, "text": chunk})

milvus_client.insert(collection_name=collection_name, data=data)

In [ ]:
#Query
question = QUESTION

In [ ]:
#Search for the question in the collection and retrieve the semantic top-6 matches.
search_res = milvus_client.search(
    collection_name=collection_name,
    data=[emb_text(question)],
    limit=6,
    search_params={"metric_type": "IP", "params": {}},
    output_fields=["text"],
)

In [ ]:
retrieved_lines_with_distances = [
    (res["entity"]["text"], res["distance"]) for res in search_res[0]
]
print(json.dumps(retrieved_lines_with_distances, indent=4))

In [ ]:
# Build context and generate response
import os
import google.generativeai as genai
from dotenv import load_dotenv

# Configure GEMINI 2.5 Flash Lite
load_dotenv()
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
genai.configure()
model = genai.GenerativeModel('gemini-2.5-flash-lite')

context = "\n".join(
    [line_with_distance[0] for line_with_distance in retrieved_lines_with_distances]
)

SYSTEM_PROMPT = """
Human: You are an AI assistant. You are able to find answers to the questions from the contextual passage snippets provided.
"""
USER_PROMPT = f"""
Use the following pieces of information enclosed in <context> tags to provide an answer to the question enclosed in <question> tags.

<context>
{context}
</context>

<question>
{question}
</question>
"""

response = model.generate_content([
    f"System: {SYSTEM_PROMPT}\n\nUser: {USER_PROMPT}"
])
print(response.text)

### Form questions (`excel` source)

In [ ]:
questions = [
            #Page 1 questions
            "Language (English/Hindi)"
            "Authorised capital of the company as on the date of filling (in Rs.)",
            "1.(a) Corporate Identity Number (CIN) of company",
            "2.(a) Name of the company",
            "2.(b) Address of the registered office of the company",
            "2.(c) e-mail ID of the company",
            "3. Financial year to which financial statement relates - From:____ (DD/MM/YYYY) To:____ (DD/MM/YYYY)",
            "4.(a) Date of Board of Director's meeting in which financial statements are approved - ____ (DD/MM/YYYY)",
            "4.(b) Nature of financial statements: ____? 4.(c) Whether provisional financial statements filed earlier (Yes/No/Not appicable). 4.(d) Whether adopted in adjourned AGM (Yes/No/Not appicable)",
            "5.(a) Whether Annual General Meeting (AGM) held (Yes/No/Not appicable), 5.(b) if YES then date of AGM:____ (DD/MM/YYYY). 5.(c) Due date of AGM:____ (DD/MM/YYYY). 5.(d) Whether any extension for financial year or AGM granted (Yes/No)",
            "6.(a) Whether Schedule 3 of the Companies Act, 2013 is applicable (Yes/No). 6.(b) Whether financial statements have been drawn on the basis of ____ (AS Taxonomy/Ind-AS Taxonomy)",

            #Page 2 questions
            "7. Type of Industry",
            "8. Whether consolidated Financial State,emts are also being filed (Yes/No)",
            "9.(a) In case if a government company, whether CAG of India has commented upon of supplemented the audit report under section 143 of the Companies Act, 2013 (Yes/No)",
            "9.(d) whether CAG of India has conducted supplementary or test audit under section 143 (Yes/No)",
            "10. Whether Secretarial Audit is applicable (Yes/No)"
            "11. Whether detailed discloure with respect to Director's report Sec 134(3) is attached (Yes/No)"
    ]

In [ ]:
if RUN_FORM_QUESTIONS:
    #create the vector db for absulutely all ExampleCo Documents in ExampleCo.zip
    milvus_client = MilvusClient(uri="./milvus_ExampleCo_all.db")
    collection_name = "rag_ExampleCo_all"

    data = []
    if milvus_client.has_collection(collection_name):
        milvus_client.drop_collection(collection_name)

    milvus_client.create_collection(
        collection_name=collection_name,
        dimension=embedding_dim,
        metric_type="IP",  # Inner product distance
        consistency_level="Strong",  # Supported values are (`"Strong"`, `"Session"`, `"Bounded"`, `"Eventually"`). See https://milvus.io/docs/consistency.md#Consistency-Level for more details.
    )
    for i, chunk in enumerate(tqdm(texts, desc="Processing chunks")):
        embedding = emb_text(chunk)
        data.append({"id": i, "vector": embedding, "text": chunk})

    milvus_client.insert(collection_name=collection_name, data=data)

In [ ]:
#iterate through all the queries 
if RUN_FORM_QUESTIONS:
    for question in questions: 
        search_res = milvus_client.search(
        collection_name=collection_name,
        data=[emb_text(question)],
        limit=6,
        search_params={"metric_type": "IP", "params": {}},
        output_fields=["text"],
    )
